# Análisis Univariante: Gráficos de Distribución de Cadmio (Cd)

Histogramas, boxplots integrados con límite permisible OMS / ATSDR (1.0 µg/L) y gráficos Q-Q a 300 DPI.

> Protocolo bioestadístico y epidemiológico de biomonitoreo con **HeavyStats** v0.4.5.

---
## 1. Carga de Datos y Configuración del Motor Gráfico (Cadmio)

Cargamos la muestra analítica de Cadmio y configuramos `UnivariatePlots` con paleta científica de alta legibilidad editorial.


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import heavystats as hs
from heavystats.univariate import UnivariatePlots

base_dir = Path.cwd() if (Path.cwd() / "datos").exists() else Path.cwd().parent
ruta_metal = base_dir / "datos" / "procesados" / "datos_cadmio_n20.csv"

if not ruta_metal.exists():
    df_raw = hs.load_data(base_dir / "datos" / "datos_cadmio.csv")
    df_analytical = hs.select_metal(hs.standardize_boolean_columns(df_raw), concentration_col="cadmio")
else:
    df_analytical = pd.read_csv(ruta_metal, encoding="utf-8")

uplots = UnivariatePlots(df_analytical, palette="crest")
salidas_graficos = base_dir / "salidas" / "univariante" / "graficos"
salidas_graficos.mkdir(parents=True, exist_ok=True)

print(f"Motor UnivariatePlots listo para generación de figuras de Cadmio a 300 DPI.")


---
## 2. Pipeline de Distribución: Boxplot e Histograma de Cadmio con Límite OMS / ATSDR (1.0 µg/L) y Máscara de Percentiles

Visualización integrada en dos paneles coordinados que combina la dispersión individual (jitter points), cuartiles, mediana, histograma con KDE, umbral de referencia toxicológico (1.0 µg/L) y **máscara de estratificación por percentiles** ($\leq P_{25}$, $> P_{25} \text{ y } < P_{75}$, $\geq P_{75}$) con bandas sombreadas, líneas verticales y conteo de muestras con porcentajes.


In [ ]:
# Pipeline de Distribución con límite permisible, escala natural y máscara de percentiles
figs_metal = uplots.plot_distribucion(
    columns="Cadmio_ug_L",
    percentile_mask=True,
    show_limit=True,
    log_scale=False,
    color="#2b6cb0",
    save_dir=str(salidas_graficos),
    save_format=["png", "pdf"],
    shrink=0.8,
    bins=5,
    dpi=300
)
plt.show()

# Resumen analítico de los estratos percentilares de Cadmio
mascara_info = uplots.obtener_mascara_percentiles(column="Cadmio_ug_L")
print("Estratificación por percentiles (<=P25, P25-P75, >=P75):")
print(f"  P25 = {mascara_info['p_lower']:.3f}, P75 = {mascara_info['p_upper']:.3f}")
for estrato, count in mascara_info['counts'].items():
    pct = mascara_info['percentages'][estrato]
    print(f"  {estrato}: n={count} ({pct:.1f}%)")


---
## 3. Histograma Estratificado con KDE por Factor de Exposición (Cadmio)

Visualización de la distribución sérica de Cadmio estratificada por covariables clave (como Sexo), utilizando barras apiladas (`multiple="stack"`), separación visual (`shrink=0.8`) y curvas de densidad KDE superpuestas para contrastar subpoblaciones.


In [ ]:
# Histograma estratificado por Sexo con barras apiladas y estimación de densidad KDE
figs_hist_sexo = uplots.plot_histograma_estratificado(
    columns="Cadmio_ug_L",
    hue="Sexo",
    multiple="stack",
    shrink=0.8,
    kde=True,
    palette="Set1",
    save_dir=str(salidas_graficos),
    save_format=["png", "pdf"],
    dpi=300
)
plt.show()


---
## 4. Gráfico Q-Q Normal de Cadmio

Evaluación visual del ajuste de la distribución observada frente a los cuantiles teóricos de la distribución Normal.


In [ ]:
# Gráficos Q-Q para Cadmio
figs_qq = uplots.plot_qq(
    columns="Cadmio_ug_L",
    save_dir=str(salidas_graficos),
    save_format=["png", "pdf"],
    dpi=300
)
plt.show()


---
## 5. Distribución de Variables Categóricas

Diagramas de barras horizontales y verticales con frecuencias absolutas y relativas para las covariables sociodemográficas.


In [ ]:
# Visualización de covariables categóricas
figs_cat = uplots.plot_categorico(
    columns=["Sexo", "Sector", "Institucion"],
    save_dir=str(salidas_graficos),
    save_format=["png", "pdf"],
    dpi=300
)
plt.show()


---
## 6. Pipeline Multirrespuesta: Fuentes de Agua

Frecuencia y porcentaje de acceso a fuentes de agua para consumo en la población escolar y comunitaria.


In [ ]:
# Gráfico de selección múltiple para fuentes de agua
if "Salud_Agua" in df_analytical.columns:
    fig_agua = uplots.plot_multirrespuesta(
        "Salud_Agua",
        title="Fuentes de Agua para Consumo Humano",
        save_dir=str(salidas_graficos),
        save_format=["png", "pdf"],
        dpi=300
    )
    plt.show()


---
## 7. Pipeline de Factores de Exposición Agrupados

Factores ambientales y ocupacionales agrupados por dimensiones analíticas (talleres, industrias, lugares de riesgo).


In [ ]:
# Factores de exposición agrupados
fig_factores = uplots.plot_factores(
    dimensions={
        "Exposicion_Lugares": "Lugares de Riesgo",
        "Exposicion_Talleres": "Talleres y Servicios",
        "Exposicion_Industrias": "Industrias Químicas/Metales"
    },
    save_dir=str(salidas_graficos),
    save_format=["png", "pdf"],
    dpi=300
)
plt.show()


---
## 8. Pipeline Mosaico de Determinantes de Riesgo

Mosaico multipanel que integra el consumo de pescado/mariscos, bombillos ahorradores y convivencia con fumadores.


In [ ]:
# Mosaico de determinantes toxicológicos mediante mosaico y plot_categorico
fig_determinantes, axd = uplots.crear_mosaico(
    [["pescado", "pescado"],
     ["bombillos", "fumadores"]],
    figsize=(10.5, 7.2),
    height_ratios=(1.25, 0.95),
    bold=True
)

uplots.plot_categorico(
    "Alim_Pescados",
    ax=axd["pescado"],
    title="(a) Consumo de Pescado y Mariscos",
    color="#2b6cb0"
)
uplots.plot_categorico(
    "Salud_Bombillos",
    ax=axd["bombillos"],
    title="(b) Bombillos Ahorradores en el Hogar",
    order=["No", "Sí"],
    color="#319795"
)
uplots.plot_categorico(
    "Salud_Fuma",
    ax=axd["fumadores"],
    title="(c) Convivencia con Fumadores en el Hogar",
    order=["No", "Sí"],
    color="#4a5568"
)

fig_determinantes.tight_layout()
plt.show()

print(f"¡Figuras univariantes guardadas exitosamente en: {salidas_graficos}!")
